# CO2 Basin Geographic Center Analysis

Calculate the geographic center (centroid) of each of the 24 CO2 injection basins in China from the DSA average injection rate raster.

In [ ]:
import rasterio
import numpy as np
from scipy import ndimage
from pyproj import Transformer
import pandas as pd

TIF_PATH = "DSA-injection-AVG-Recommended.tif"

with rasterio.open(TIF_PATH) as src:
    data = src.read(1)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

print(f"Raster size: {src.width} x {src.height}")
print(f"CRS: {crs.to_string()[:80]}...")
print(f"Bounds: {src.bounds}")

In [ ]:
# Identify connected regions (basins) via connected-component labeling
mask = (data != nodata) & np.isfinite(data)
labeled, n_components = ndimage.label(mask)
print(f"Total connected components found: {n_components}")

# Sort by pixel count descending; take top 24 as the 24 basins
counts = sorted(
    [(label, int(np.sum(labeled == label))) for label in range(1, n_components + 1)],
    key=lambda x: -x[1],
)
top24 = counts[:24]

print(f"\nTop 24 components range: {top24[0][1]} – {top24[-1][1]} pixels")
print(f"Rank-25 component size for reference: {counts[24][1]} pixels")

In [ ]:
# Compute centroids in projected coordinates, then convert to WGS84 lon/lat
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

records = []
for rank, (label, n_pixels) in enumerate(top24, 1):
    rows, cols = np.where(labeled == label)
    # Pixel-center projected coordinates via affine transform
    xs = transform.c + (cols + 0.5) * transform.a
    ys = transform.f + (rows + 0.5) * transform.e
    lon, lat = transformer.transform(xs.mean(), ys.mean())
    records.append({"basin": rank, "pixels": n_pixels, "lon": lon, "lat": lat})

basin_df = pd.DataFrame(records)
basin_df["lon"] = basin_df["lon"].round(4)
basin_df["lat"] = basin_df["lat"].round(4)
basin_df

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(
    figsize=(12, 8),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# Focus on China + surrounding region
ax.set_extent([70, 140, 15, 55], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor="#f5f5f0", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="#d0e8f5", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle="--", zorder=1)
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)

# Plot basin centroids; scale marker size by pixel count
sizes = basin_df["pixels"] / basin_df["pixels"].max() * 300 + 30
sc = ax.scatter(
    basin_df["lon"],
    basin_df["lat"],
    s=sizes,
    c=basin_df["pixels"],
    cmap="YlOrRd",
    edgecolors="k",
    linewidths=0.5,
    transform=ccrs.PlateCarree(),
    zorder=2,
)
plt.colorbar(sc, ax=ax, label="Basin area (pixels)", shrink=0.6)

# Label each basin
for _, row in basin_df.iterrows():
    ax.text(
        row["lon"] + 0.5,
        row["lat"] + 0.5,
        str(int(row["basin"])),
        fontsize=7,
        transform=ccrs.PlateCarree(),
        zorder=3,
    )

ax.set_title("Geographic Centers of 24 CO\u2082 Injection Basins in China", fontsize=13)
plt.tight_layout()
plt.savefig("basin_centroids_map.png", dpi=150, bbox_inches="tight")
plt.show()

---
## CO2 Storage Potential Analysis

Same 24 basins (identical connected-component structure), now using `DSA-storage potential.tif` to compute per-basin storage potential statistics.

In [ ]:
STORAGE_TIF = "DSA-storage potential.tif"

with rasterio.open(STORAGE_TIF) as src_s:
    storage_data = src_s.read(1)
    storage_nodata = src_s.nodata

# Basins are identical to the injection raster; reuse `labeled` and `top24` from above.
storage_records = []
for rank, (label, n_pixels) in enumerate(top24, 1):
    vals = storage_data[labeled == label]
    valid = vals[(vals != storage_nodata) & np.isfinite(vals)]
    storage_records.append({
        "basin": rank,
        "lon": basin_df.loc[basin_df["basin"] == rank, "lon"].values[0],
        "lat": basin_df.loc[basin_df["basin"] == rank, "lat"].values[0],
        "mean_storage": round(float(valid.mean()), 4),
        "total_storage": round(float(valid.sum()), 2),
        "min_storage": round(float(valid.min()), 4),
        "max_storage": round(float(valid.max()), 4),
    })

storage_df = pd.DataFrame(storage_records)
storage_df

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(18, 7),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

def base_map(ax):
    ax.set_extent([70, 140, 15, 55], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#f5f5f0", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="#d0e8f5", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle="--", zorder=1)
    ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)

# --- Left: mean storage potential ---
ax = axes[0]
base_map(ax)
sizes = storage_df["mean_storage"] / storage_df["mean_storage"].max() * 300 + 30
sc = ax.scatter(
    storage_df["lon"], storage_df["lat"],
    s=sizes, c=storage_df["mean_storage"],
    cmap="Blues", edgecolors="k", linewidths=0.5,
    transform=ccrs.PlateCarree(), zorder=2,
)
plt.colorbar(sc, ax=ax, label="Mean storage potential", shrink=0.6)
for _, row in storage_df.iterrows():
    ax.text(row["lon"] + 0.5, row["lat"] + 0.5, str(int(row["basin"])),
            fontsize=7, transform=ccrs.PlateCarree(), zorder=3)
ax.set_title("Mean CO₂ Storage Potential per Basin", fontsize=12)

# --- Right: total storage potential ---
ax = axes[1]
base_map(ax)
sizes = storage_df["total_storage"] / storage_df["total_storage"].max() * 300 + 30
sc = ax.scatter(
    storage_df["lon"], storage_df["lat"],
    s=sizes, c=storage_df["total_storage"],
    cmap="Greens", edgecolors="k", linewidths=0.5,
    transform=ccrs.PlateCarree(), zorder=2,
)
plt.colorbar(sc, ax=ax, label="Total storage potential", shrink=0.6)
for _, row in storage_df.iterrows():
    ax.text(row["lon"] + 0.5, row["lat"] + 0.5, str(int(row["basin"])),
            fontsize=7, transform=ccrs.PlateCarree(), zorder=3)
ax.set_title("Total CO₂ Storage Potential per Basin", fontsize=12)

plt.suptitle("CO₂ Storage Potential Across 24 China Basins", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("basin_storage_map.png", dpi=150, bbox_inches="tight")
plt.show()